In [6]:
%pip install keras-hub
%pip install matplotlib pillow requests

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import urllib.request
import requests
import numpy as np
import tensorflow as tf
import keras_hub
import matplotlib.pyplot as plt
from PIL import Image
import ssl


ssl._create_default_https_context = ssl._create_unverified_context
# 1. Download and Load ImageNet Class Names
print("Fetching ImageNet class labels...")
url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
response = requests.get(url)
imagenet_classes = response.text.strip().split("\n")

# 2. Load the mandatory KerasHub components (vit_base_patch16_224_imagenet preset)
print("Loading ViT Preprocessor...")
preprocessor = keras_hub.models.ViTImageClassifierPreprocessor.from_preset(
    "vit_base_patch16_224_imagenet"
)

print("Loading Pretrained Vision Transformer (ViT) Model...")
classifier = keras_hub.models.ViTImageClassifier.from_preset(
    "vit_base_patch16_224_imagenet", 
    preprocessor=preprocessor
)

# 3. Define the URLs for 5 distinct real-world ImageNet classes
image_urls = {
    "lion.jpg": "https://images.unsplash.com/photo-1546182990-dffeafbe841d?auto=format&fit=crop&w=400&q=80",
    "sports_car.jpg": "https://images.unsplash.com/photo-1503376780353-7e6692767b70",
    "golden_retriever.jpg": "https://images.unsplash.com/photo-1552053831-71594a27632d",
    "tabby.jpg": "https://images.unsplash.com/photo-1518791841217-8f162f1e1131?auto=format&fit=crop&w=400&q=80",
    "espresso.jpg": "https://images.unsplash.com/photo-1511920170033-f8396924c348",
}

# Download the images locally using a proper browser User-Agent
for filename, download_url in image_urls.items():
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        try:
            # Create a request object with standard browser headers to bypass the 403 block
            req = urllib.request.Request(
                download_url, 
                headers={'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'}
            )
            with urllib.request.urlopen(req) as response, open(filename, 'wb') as out_file:
                out_file.write(response.read())
        except Exception as e:
            print(f"Failed to download {filename}. Error: {e}")

# 4. Process and Predict Classes for each image
print("\n--- Running Predictions ---")
results = []

for filename in image_urls.keys():
    # Load raw image via PIL
    raw_img = Image.open(filename).convert("RGB")
    
    # Convert image to numpy array structure matching expected tensor formats
    img_array = tf.keras.utils.img_to_array(raw_img)
    
    # Add a batch dimension -> shape becomes (1, height, width, 3)
    batched_img = np.expand_dims(img_array, axis=0)
    
    # Generate predictions (the preprocessor handles sizing & scaling behind the scenes)
    predictions = classifier.predict(batched_img)
    
    # Find the index with the highest probability value
    predicted_class_idx = np.argmax(predictions, axis=-1)[0]
    predicted_label = imagenet_classes[predicted_class_idx]
    
    print(f"File: {filename} | Predicted: {predicted_label}")
    results.append((filename, predicted_label))

2026-05-17 00:42:14.964712: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching ImageNet class labels...
Loading ViT Preprocessor...
Loading Pretrained Vision Transformer (ViT) Model...

--- Running Predictions ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
File: lion.jpg | Predicted: lion
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 887ms/step
File: sports_car.jpg | Predicted: racer
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
File: golden_retriever.jpg | Predicted: golden retriever
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
File: tabby.jpg | Predicted: tabby
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
File: espresso.jpg | Predicted: espresso


| Image | File | Predicted Label | True Label | Correct? |
| :--- | :----: | ---: | ---: | ---: |
| Image  | 1 | lion | lion | Yes |
| Image  | 2 | racer | sports car | Yes |
| Image  | 3 | golden retreiver | golden retreiver | Yes |
| Image  | 4 | tabby | cat | Yes |
| Image  | 5 | espresso | espresso | Yes |

The pretrained ViT model successfully classified the 5 real-world images. In some cases, it predicted a more specific subclass, such as a dog breed, instead of a general class name, which is still considered correct.